# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**My lane: Refresh / Content Opportunity Scoring.** The question is *"which pages should an editor look at first?"* — and that makes it a **ranking / scoring** task. The output is one priority score per page, sorted into a review queue.

- **Why not a yes/no classifier alone?** An editor can only fix so many pages a week. A yes/no label says "this page is in trouble" for over half the pages (see below) — it doesn't say *which first*. The score gives the order.
- **Why not clustering?** Clustering would tell me what *kinds* of pages exist, but not which one needs attention now.
- A classifier can still sit *inside* the scoring (its predicted probability becomes the score), but the thing I'm judged on is the **order of the queue**.

In [1]:
import os, sys, subprocess
import pandas as pd, numpy as np

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/stanley-metone/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
assert df.shape == (30000, 44), "unexpected starter file shape"
print("Lane: Refresh / Content Opportunity Scoring  ->  task type: ranking / scoring")
print("Starter slice loaded:", df.shape[0], "pages x", df.shape[1], "columns,", df["client_id"].nunique(), "pseudonymized clients")

Lane: Refresh / Content Opportunity Scoring  ->  task type: ranking / scoring
Starter slice loaded: 30000 pages x 44 columns, 32 pseudonymized clients


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target (a proxy):** `is_declining` = 1 when a page's search impressions in the last 30 days fell by more than 20% compared with the 30 days before (`trend_direction == "down"` in the starter file), otherwise 0.

**Where does the label come from — observed or defined?** Both, and I want to be honest about it. The impressions are *observed* measurements. But the "more than 20%" line is a *rule someone chose*, so this is a **proxy** for "this page needs attention", not a directly observed outcome. If I'm careless, a model will just re-learn the 20% rule instead of learning anything about the world.

**What that means for features:**
- `trend_direction` and `trend_pct` are **never features** — they are the label (the starter's data dictionary says the same).
- The label is also built from `impressions_last_30d` and `impressions_prev_30d`, so I will treat those as suspect too and check them in the week-3 leakage assignment before using them.

**A better target later (sketch, not built yet):** with the warehouse's daily table I can pick a cut-off date, build features only from days *before* it, and measure the outcome in the 30 days *after* it (for example, did clicks keep falling?). That is an observed outcome in a later window — better than a rule-defined label. This week I only sketch it.

In [2]:
df["is_declining"] = (df["trend_direction"] == "down").astype(int)

print("Share of pages labelled declining:", round(df["is_declining"].mean(), 4))
print()
print(df["trend_direction"].value_counts().rename("pages").to_frame())

LABEL_SOURCES = ["trend_direction", "trend_pct"]            # never features
SUSPECT_UNTIL_CHECKED = ["impressions_last_30d", "impressions_prev_30d"]  # label is computed from these
print("\nNever features:", LABEL_SOURCES, "| suspect until leakage check:", SUSPECT_UNTIL_CHECKED)

# what the target column looks like next to the numbers it comes from
df[["content_type", "impressions_prev_30d", "impressions_last_30d", "trend_pct", "trend_direction", "is_declining"]].sample(8, random_state=42)

Share of pages labelled declining: 0.5421

                 pages
trend_direction       
down             16262
stable            5962
up                4388
new               2236
flat              1152

Never features: ['trend_direction', 'trend_pct'] | suspect until leakage check: ['impressions_last_30d', 'impressions_prev_30d']


,content_type,impressions_prev_30d,impressions_last_30d,trend_pct,trend_direction,is_declining
2308,keyword article,50,54,8.0,stable,0
22404,keyword article,3172,3897,22.9,up,0
23397,feedly article,0,3,NaN,new,0
25058,comparison article,61,26,-57.4,down,1
2664,keyword article,1461,1157,-20.8,down,1
8511,keyword article,218,341,56.4,up,0
5148,keyword article,2269,2608,14.9,stable,0
7790,keyword article,4,1,-75.0,down,1


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Success metric: Precision@50, compared with the base rate.** Take the 50 pages my score puts at the top of the queue; Precision@50 is the share of them that really are declining.

- **Why 50?** It is a stand-in for how many pages one editor could realistically review in a week. That is my *assumption* — I will check it against what FlyRank's editors can actually handle. (The starter pipeline also reports Precision@50.)
- **Why compare with the base rate?** Over half of all pages are declining: 54% of all 30,000, and about 60% of the pages I would actually rank (the ones with enough traffic, see section 4). So a *random* queue is already right 54–60% of the time. A model scoring 62% would sound good and be almost worthless. "Good" means **clearly above both the base rate and the best plain rule, at the same K, on clients the model never saw** (grouped by `client_id`).
- I am deliberately not inventing a target number now. The week-4 baseline will tell me what the best plain rule scores, and "good" is whatever beats it.

In [3]:
def precision_at_k(y_true, score, k=50):
    """Share of the k highest-scored pages that are truly declining."""
    y_true = np.asarray(y_true); score = np.asarray(score)
    top = np.argsort(-score, kind="stable")[:k]
    return y_true[top].mean()

# the queue only contains pages with enough traffic to read a trend from (explained in section 4)
df["measurable"] = ((df["impressions_90d"] >= 100) & (df["sessions_90d"] > 0)).astype(int)
queue = df[df["measurable"] == 1]

rng = np.random.default_rng(0)
random_p50 = np.mean([precision_at_k(queue["is_declining"], rng.random(len(queue)), 50) for _ in range(500)])

print(f"Base rate, all pages:                         {df['is_declining'].mean():.3f}")
print(f"Base rate, pages I would actually rank:       {queue['is_declining'].mean():.3f}")
print(f"Random queue, Precision@50 (avg of 500 draws): {random_p50:.3f}   <- the floor any score must clearly beat")

Base rate, all pages:                         0.542
Base rate, pages I would actually rank:       0.598
Random queue, Precision@50 (avg of 500 draws): 0.595   <- the floor any score must clearly beat


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = one content item (page).** In the starter file each row is one pseudonymized page, with its metrics summed over a trailing 90-day window. The queue I want to rank has one row per page, so this is already the right grain.

Two choices I'm making about *which* rows count:
- The queue should only include pages with enough traffic to read a trend from (`impressions_90d >= 100` and `sessions_90d > 0`, the starter's own `measurable_opportunity` idea). A page with 3 impressions "dropping 40%" is noise.
- Pages from the same client are not independent, so later I will split train/test **by client**, never by random page.

In [4]:
assert df["content_id"].is_unique, "grain check failed: one row should be exactly one page"

unit = df[df["measurable"] == 1]
print("Grain check passed: one row = one page (content_id is unique).")
print(f"Pages in the starter slice: {len(df):,} | measurable (enough traffic to rank): {len(unit):,}")
print(f"Declining share among measurable pages: {unit['is_declining'].mean():.3f}")
print("Pages per client (measurable): min", unit.groupby("client_id").size().min(),
      "| median", int(unit.groupby("client_id").size().median()),
      "| max", unit.groupby("client_id").size().max())

unit_cols = ["content_id", "content_type", "age_tier", "position_tier", "impressions_90d",
             "clicks_90d", "ctr", "days_since_last_update", "is_declining"]
unit[unit_cols].head(8)

Grain check passed: one row = one page (content_id is unique).
Pages in the starter slice: 30,000 | measurable (enough traffic to rank): 22,006
Declining share among measurable pages: 0.598
Pages per client (measurable): min 1 | median 253 | max 6579


,content_id,content_type,age_tier,position_tier,impressions_90d,clicks_90d,ctr,days_since_last_update,is_declining
0,content_304f48230142,keyword article,181-365,striking,3803,29,0.76,20,1
1,content_a1fb4e703a9e,keyword article,365+,page_3_5,15320,7,0.05,25,1
2,content_9aa793d4d895,keyword article,91-180,page_3_5,12581,11,0.09,20,1
3,content_331d6c4de07b,keyword article,365+,page_1,11751,58,0.49,22,0
4,content_d99b7a2d90ca,keyword article,181-365,page_3_5,19140,24,0.13,14,1
5,content_d4084a4bc775,keyword article,91-180,page_1,3970,1,0.03,20,1
7,content_a63219c6e95a,keyword article,365+,page_3_5,1724,1,0.06,22,0
8,content_5e6c160719bc,keyword article,31-90,page_3_5,32574,29,0.09,20,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

**Why ML (or at least a data-driven score) instead of one fixed rule?** A fixed rule means one or two columns and a cut-off, like "refresh the oldest pages". The cells below show why that is shaky here:

1. **The pattern depends on several columns at once.** The share declining swings between about 0.2 and 0.8 across position-tier x age-tier cells (table 1), so no single column explains it. It also depends on a choice I make: among *all* 30,000 pages, `top_3` pages decline the least (about 24%), but among pages with enough traffic they decline the *most* (table 1). A fixed rule built on one slice of the data would point the wrong way on another.
2. **A single-column rule is unstable.** "Oldest pages first" looks great in the top 50 and falls *below* the base rate by the top 500 (table 2). A rule that only works at one cut-off is not a rule I can hand an editor.
3. **Clients behave very differently.** The share declining per client runs from 0.00 to 1.00 with a median near 0.56 (output 3; the extremes include very small clients), so one global cut-off cannot fit everyone.

**What I have *not* shown:** that a model beats these rules. That is a claim for weeks 4–5, after I build the plain-rule baseline and test a model on held-out clients. For now this is a **hypothesis supported by the data, not a result**.

In [5]:
# 1) decline rate depends on several columns at once (measurable pages only)
tab = (unit.pivot_table(index="position_tier", columns="age_tier", values="is_declining", aggfunc="mean")
          .round(2))
print("Table 1 - share declining, by position tier (rows) x page age tier (columns):")
print(tab.to_string())
print()
print("\nSame position tiers on ALL 30,000 pages (no traffic filter):")
print(df.groupby("position_tier")["is_declining"].mean().round(2).to_string())
print()
print(unit.groupby("content_type")["is_declining"].agg(share_declining="mean", pages="size").round(3).to_string())

# 2) one-column rules: how good is the top of the queue at K=50 vs K=500?
rules = {"most impressions first": unit["impressions_90d"],
         "oldest pages first": unit["content_age_days"],
         "longest since last update first": unit["days_since_last_update"],
         "lowest CTR first": -unit["ctr"]}
rows = [{"rule": name,
         "precision@50": precision_at_k(unit["is_declining"], s, 50),
         "precision@500": precision_at_k(unit["is_declining"], s, 500)} for name, s in rules.items()]
out = pd.DataFrame(rows).set_index("rule").round(3)
out.loc["(base rate, measurable pages)"] = [round(unit["is_declining"].mean(), 3)] * 2
print("\nTable 2 - single-column rules (ties broken by file order):")
print(out.to_string())

# 3) clients differ a lot
by_client = unit.groupby("client_id")["is_declining"].mean()
print("\nOutput 3 - share declining per client: min %.2f | median %.2f | max %.2f"
      % (by_client.min(), by_client.median(), by_client.max()))

Table 1 - share declining, by position tier (rows) x page age tier (columns):
age_tier       181-365  31-90  365+  91-180
position_tier                              
deep              0.33   0.50  0.21    0.57
page_1            0.60   0.70  0.52    0.65
page_3_5          0.61   0.64  0.33    0.74
striking          0.63   0.69  0.47    0.71
top_3             0.71   0.83  0.69    0.80


Same position tiers on ALL 30,000 pages (no traffic filter):
position_tier
deep        0.34
page_1      0.57
page_3_5    0.56
striking    0.61
top_3       0.24

                    share_declining  pages
content_type                              
comparison article            0.650    366
feedly article                0.676    352
keyword article               0.595  21288

Table 2 - single-column rules (ties broken by file order):
                                 precision@50  precision@500
rule                                                        
most impressions first                  0.420         

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

**The action this output supports:** the ranked list becomes a weekly *refresh queue* — an editor opens the top pages, checks them, and updates or rewrites the ones that really are slipping. A wrong call costs editor time on a healthy page (false alarm) or a real decline left unfixed for another month (miss).